# CNN-BiLSTM-MHA + PPO Trader — Training (V4)

Full training pipeline for the CNN-BiLSTM-MHA (forecaster) and PPO (agent) models,
with improved Reward V2, vectorization for 1 A100 GPU, and TensorFlow.js export.

**Changes from V3 → V4:**
- **Soft labels** (sigmoid of future return) replace hard 0/1 binary targets
- **Wavelet decomposition** (db4, level-2) adds 6 frequency-domain features
  - Trend deviation, medium-freq oscillation, high-freq noise level
  - Trend slope, noise/signal energy ratio, spectral entropy
- NUM_FEATURES: 10 → 16

**Unchanged:** CNN-BiLSTM-MHA architecture, PPO agent, reward V2, vectorized runner.


In [ ]:
# Training deps only: ccxt (data) + PyWavelets (features). Both pure Python --
# they do NOT touch TensorFlow/CUDA, so Colab's GPU-matched TF stays intact.
# tensorflowjs is installed later, in the conversion cell (after all training),
# because its CUDA deps can corrupt the GPU context (CUDA_ERROR_INVALID_HANDLE).
!pip install -q ccxt PyWavelets

In [2]:
import math, time, os, json
from contextlib import nullcontext
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import ccxt

# Hyperparameters
WINDOW_SIZE          = 128            # 128 x 15min ~ 32h of context
HORIZON              = 4              # predict direction 4 x 15min = 60 min ahead
HIDDEN_UNITS         = 64
DROPOUT              = 0.2
NUM_FEATURES         = 16             # 10 original + 6 wavelet
L2                   = 1e-5
LR                   = 5e-4
MIN_LR               = 1e-5
EARLY_STOP_PATIENCE  = 15
BATCH_SIZE           = 256
FORECAST_EPOCHS      = 100

# PPO
STATE_SIZE   = 13
ACTION_SIZE  = 3       # 0=HOLD 1=BUY 2=SELL
GAMMA        = 0.99
CLIP_RATIO   = 0.2
POLICY_LR    = 3e-4
VALUE_LR     = 1e-3
PPO_MIN_LR   = 1e-5
LR_DECAY     = 0.99
MAX_GRAD_NORM = 0.5
PPO_EPOCHS   = 10

# Constants
INDICATOR_WARMUP      = 200
RETURN_CLIP           = 0.1
VOLATILITY_WINDOW     = 20
FEE_RATE              = 0.000
BACKTEST_HOLDOUT      = 1000
VALIDATION_FRACTION   = 0.1
EMBARGO_FRACTION      = 0.01
HISTORICAL_CANDLES    = 35_000        # 35k x 15min ~ 365 days

# Risk - matches .env values used at inference. Stop/take based on 15m ATR.
STOP_LOSS_PCT    = 0.0030             # 0.30%
TAKE_PROFIT_PCT  = 0.0050             # 0.50%, R:R 1.67
SLIPPAGE_PCT     = 0.0005

# SOFT-LABEL CLASSIFICATION MODE:
# Labels are no longer hard 0/1. Instead, each label is a soft probability
# proportional to the magnitude of the future return, via sigmoid transform.
# A +0.001% move → label ≈ 0.52 (nearly uncertain)
# A +2.0%  move → label ≈ 0.98 (very confident bullish)
# A -1.5%  move → label ≈ 0.05 (very confident bearish)
# This gives the model richer gradient signal on strong moves
# while naturally down-weighting ambiguous noise near zero.
SOFT_LABEL_SCALE     = 150            # sigmoid(excess_return * SCALE): graded labels, avoid saturation/collapse
SOFT_LABEL_CLIP      = (0.02, 0.98)   # avoid extremes for numeric stability
#
# Targets are soft floats in [0.02, 0.98] (sigmoid of future return).
# pred_ret in reward function is P(up) in [0,1].
# pred_strength = (pred_ret - 0.5) is the signed directional confidence.
PRED_STRENGTH_OPPORTUNITY_TH = 0.05   # P(up) > 0.55 = strong enough to regret HOLD
LOSS_CUT_THRESHOLD = STOP_LOSS_PCT / 3.0  # unchanged: cut loss bonus on realized PnL

FORECASTER_SYMBOLS = [
    "BTC/USDT", "ETH/USDT",
    "SOL/USDT", "LINK/USDT", "BNB/USDT",
    "XRP/USDT", "DOGE/USDT"
]
AGENT_SYMBOL       = "BTC/USDT"
TIMEFRAME          = "15m"

# Multi-GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)} -> {[g.name for g in gpus]}")

if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
else:
    strategy = tf.distribute.get_strategy()

NUM_REPLICAS = strategy.num_replicas_in_sync
GLOBAL_BATCH = BATCH_SIZE * NUM_REPLICAS
print(f"Replicas: {NUM_REPLICAS}, global batch: {GLOBAL_BATCH}")

# On a single device the default-strategy scope can corrupt the CUDA stream on
# some Colab TF builds (CUDA_ERROR_INVALID_HANDLE during model build). It adds
# nothing with one GPU, so use a no-op context unless we truly have >1 replica.
def dist_scope():
    return strategy.scope() if strategy.num_replicas_in_sync > 1 else nullcontext()
import os

# Portable working dir: Kaggle, Colab, or local. All outputs go under WORK_DIR.
if os.path.isdir("/kaggle/working"):
    WORK_DIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK_DIR = "/content"
else:
    WORK_DIR = "."
print(f"WORK_DIR = {WORK_DIR}")


2026-05-21 21:20:24.619829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779398424.866335      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779398424.932531      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779398425.454653      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779398425.454700      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779398425.454703      22 computation_placer.cc:177] computation placer alr

GPUs detected: 2 -> ['/physical_device:GPU:0', '/physical_device:GPU:1']
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


Replicas: 2, global batch: 512


I0000 00:00:1779398455.599215      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779398455.605488      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [ ]:
# ── GPU SMOKE TEST ───────────────────────────────────────────────────────────
# Minimal check that the CUDA context is healthy BEFORE building the model.
# If the cast below raises CUDA_ERROR_INVALID_HANDLE, the problem is the runtime
# environment (TF/CUDA mismatch or a corrupted context), NOT the model code:
#   1) Runtime > Disconnect and delete runtime (a plain Restart is not enough),
#   2) reconnect with a GPU, run this cell FIRST (before any extra pip install).
#   3) if it passes here but fails after the install cell, the install is the cause.
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
print("Strategy replicas:", strategy.num_replicas_in_sync if "strategy" in globals() else "(run the setup cell above first)")
with tf.device('/GPU:0'):
    _probe = tf.cast(tf.constant([1.0, 2.0]), tf.float32)
    print("GPU cast OK:", _probe.numpy())

In [3]:
# ── Technical indicators (pure NumPy) ────────────────────────────────────────

def ema(values: np.ndarray, period: int) -> np.ndarray:
    alpha = 2.0 / (period + 1)
    result = np.full(len(values), np.nan)
    if len(values) < period:
        return result
    result[period - 1] = np.mean(values[:period])
    for i in range(period, len(values)):
        result[i] = values[i] * alpha + result[i - 1] * (1 - alpha)
    return result

def rsi(values: np.ndarray, period: int = 14) -> np.ndarray:
    deltas = np.diff(values)
    result = np.full(len(values), np.nan)
    if len(deltas) < period:
        return result
    gains = np.where(deltas > 0, deltas, 0.0)
    losses = np.where(deltas < 0, -deltas, 0.0)
    avg_g, avg_l = np.mean(gains[:period]), np.mean(losses[:period])
    rs = avg_g / (avg_l + 1e-10)
    result[period] = 100 - 100 / (1 + rs)
    for i in range(period, len(deltas)):
        avg_g = (avg_g * (period - 1) + gains[i]) / period
        avg_l = (avg_l * (period - 1) + losses[i]) / period
        rs = avg_g / (avg_l + 1e-10)
        result[i + 1] = 100 - 100 / (1 + rs)
    return result

def macd(values: np.ndarray, fast=12, slow=26, signal=9):
    fast_ema = ema(values, fast)
    slow_ema = ema(values, slow)
    macd_line = fast_ema - slow_ema
    signal_line = ema(np.where(np.isnan(macd_line), 0, macd_line), signal)
    return macd_line, signal_line

def bollinger(values: np.ndarray, period: int = 20, std_dev: float = 2.0):
    upper = np.full(len(values), np.nan)
    lower = np.full(len(values), np.nan)
    for i in range(period - 1, len(values)):
        window = values[i - period + 1 : i + 1]
        m = np.mean(window)
        s = np.std(window, ddof=0)
        upper[i] = m + std_dev * s
        lower[i] = m - std_dev * s
    return upper, lower

print("Indicators OK.")

# ── Wavelet decomposition (dual-path frequency features) ─────────────────────
# Decomposes close prices into frequency bands using Discrete Wavelet Transform.
# Level-2 DWT produces: cA2 (trend), cD2 (medium-freq), cD1 (high-freq/noise).
# This separates the signal into components that capture:
#   - cA2: underlying trend (denoised price direction)
#   - cD2: medium-frequency cycles (multi-bar momentum patterns)
#   - cD1: high-frequency noise (tick-level volatility, market microstructure)
#
# We use Daubechies-4 wavelet ('db4') which is standard for financial signals.

import pywt
import numpy as np

WAVELET_NAME  = 'db4'
WAVELET_LEVEL = 2

def wavelet_decompose_series(closes: np.ndarray) -> dict:
    """Decompose full close series into frequency bands.
    Returns dict with arrays aligned to original length."""
    n = len(closes)

    # Pad to nearest power of 2 for clean DWT
    pad_len = int(2 ** np.ceil(np.log2(n)))
    padded = np.pad(closes, (0, pad_len - n), mode='edge')

    # Multilevel DWT: returns [cA2, cD2, cD1]
    coeffs = pywt.wavedec(padded, WAVELET_NAME, level=WAVELET_LEVEL)

    # Reconstruct each band at full resolution
    # Trend (low-freq): zero out detail coefficients
    trend_coeffs = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    trend = pywt.waverec(trend_coeffs, WAVELET_NAME)[:n]

    # Medium-freq: zero out approximation and high-freq detail
    med_coeffs = [np.zeros_like(coeffs[0]), coeffs[1]] + \
                 [np.zeros_like(c) for c in coeffs[2:]]
    medium = pywt.waverec(med_coeffs, WAVELET_NAME)[:n]

    # High-freq: zero out everything except finest detail
    high_coeffs = [np.zeros_like(coeffs[0])] + \
                  [np.zeros_like(c) for c in coeffs[1:-1]] + [coeffs[-1]]
    high = pywt.waverec(high_coeffs, WAVELET_NAME)[:n]

    return {'trend': trend, 'medium': medium, 'high': high}


def wavelet_energy_ratio(detail: np.ndarray, window: int = 20) -> np.ndarray:
    """Rolling energy (squared magnitude) of a wavelet band."""
    energy = np.zeros(len(detail))
    for i in range(window, len(detail)):
        segment = detail[i - window : i]
        energy[i] = np.mean(segment ** 2)
    return energy


def wavelet_trend_slope(trend: np.ndarray, window: int = 10) -> np.ndarray:
    """Rolling slope of the wavelet trend (direction + strength)."""
    slope = np.zeros(len(trend))
    for i in range(window, len(trend)):
        segment = trend[i - window : i]
        x = np.arange(window)
        slope[i] = np.polyfit(x, segment, 1)[0]
    return slope


def spectral_entropy_rolling(closes: np.ndarray, window: int = 32) -> np.ndarray:
    """Rolling spectral entropy — measures how 'complex' the frequency content is.
    High entropy = noisy/chaotic, low entropy = dominated by few frequencies."""
    entropy = np.zeros(len(closes))
    for i in range(window, len(closes)):
        segment = closes[i - window : i]
        fft_amp = np.abs(np.fft.rfft(segment - np.mean(segment)))
        psd = fft_amp ** 2
        psd_norm = psd / (np.sum(psd) + 1e-12)
        entropy[i] = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))
    # Normalize to [0, 1]
    max_ent = np.log2(window // 2 + 1)  # max possible entropy
    entropy = entropy / max_ent
    return entropy

print("Wavelet decomposition + spectral entropy OK.")


Indicators OK.
Wavelet decomposition + spectral entropy OK.


In [4]:
# ── Candle fetch — multi-exchange with fallback ───────────────────────────────

TIMEFRAME_MS = {
    "1m": 60_000, "3m": 180_000, "5m": 300_000, "15m": 900_000,
    "30m": 1_800_000, "1h": 3_600_000, "2h": 7_200_000, "4h": 14_400_000,
    "6h": 21_600_000, "8h": 28_800_000, "12h": 43_200_000, "1d": 86_400_000,
}
MAX_PER_REQUEST = 1000

EXCHANGE_CONFIGS = [
    (lambda: ccxt.bybit({"enableRateLimit": True}),  {"category": "spot"}),
    (lambda: ccxt.okx({"enableRateLimit": True}),    {}),
    (lambda: ccxt.kucoin({"enableRateLimit": True}), {}),
    (lambda: ccxt.gate({"enableRateLimit": True}),   {}),
    (lambda: ccxt.mexc({"enableRateLimit": True}),   {}),
]

_active_exchange = None
_active_params   = {}

def _get_exchange():
    global _active_exchange, _active_params
    for factory, extra_params in EXCHANGE_CONFIGS:
        ex = factory()
        try:
            ex.load_markets()
            if "BTC/USDT" not in ex.markets:
                print(f"  x {ex.id} -- BTC/USDT not found")
                continue
            test = ex.fetch_ohlcv("BTC/USDT", "1h", limit=3, params=extra_params)
            if not test or len(test) == 0:
                print(f"  x {ex.id} -- fetch_ohlcv returned empty")
                continue
            print(f"  v Active exchange: {ex.id} (smoke-test OK: {len(test)} candles)")
            _active_exchange = ex
            _active_params   = extra_params
            return ex, extra_params
        except Exception as e:
            print(f"  x {ex.id} unavailable: {type(e).__name__}: {str(e)[:80]}")
    raise RuntimeError("No exchange accessible. Check Kaggle network connectivity.")

print("Searching for available exchange...")
_active_exchange, _active_params = _get_exchange()

# Diagnose available pairs
for sym in FORECASTER_SYMBOLS:
    available = sym in _active_exchange.markets
    print(f"  {sym}: {'OK' if available else 'NOT FOUND'}")

def fetch_candles(symbol: str, limit: int, timeframe: str = "1h") -> np.ndarray:
    """Returns array of shape (N, 6): [ts, open, high, low, close, volume]."""
    global _active_exchange, _active_params
    ex     = _active_exchange
    params = _active_params

    if symbol not in ex.markets:
        print(f"  ! {symbol} not found in {ex.id}")
        return np.empty((0, 6), dtype=np.float64)

    tf_ms      = TIMEFRAME_MS[timeframe]
    end_time   = int(time.time() * 1000)
    start_time = end_time - limit * tf_ms
    all_rows   = []
    since      = start_time

    while since < end_time:
        try:
            raw = ex.fetch_ohlcv(symbol, timeframe, since=since,
                                  limit=MAX_PER_REQUEST, params=params)
        except Exception as e:
            print(f"  ! Error on {ex.id} ({type(e).__name__}), reconnecting...")
            ex, params = _get_exchange()
            continue

        if not raw:
            break

        filtered = [row for row in raw if row[0] <= end_time]
        all_rows.extend(filtered)

        # Advance cursor by timestamp, NOT by batch size
        next_since = raw[-1][0] + tf_ms
        if next_since <= since:
            break
        since = next_since

        if len(all_rows) >= limit:
            break

        time.sleep(ex.rateLimit / 1000)

    if len(all_rows) == 0:
        print(f"  ! WARNING: 0 candles returned for {symbol}!")
        return np.empty((0, 6), dtype=np.float64)

    arr = np.array(all_rows, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    _, idx = np.unique(arr[:, 0], return_index=True)
    arr = arr[idx]

    print(f"  -> {symbol}: {len(arr)} candles downloaded (requested: {limit})")
    return arr[-limit:]

print("Fetch function ready.")


Searching for available exchange...


  x bybit unavailable: RateLimitExceeded: bybit GET https://api.bybit.com/v5/market/instruments-info?category=spot 403 For


  v Active exchange: okx (smoke-test OK: 3 candles)
  BTC/USDT: OK
  ETH/USDT: OK
  SOL/USDT: OK
  LINK/USDT: OK
  BNB/USDT: OK
  XRP/USDT: OK
  DOGE/USDT: OK
Fetch function ready.


In [5]:
# Feature builder (16 features per timestep) + SOFT direction targets
#
# Features 0-9:  original (price, volume, RSI, MACD, BB, EMAs, etc.)
# Features 10-15: wavelet dual-path
#   10: denoised_trend / close - 1     (how far price is from its trend)
#   11: medium_band / close            (medium-freq oscillation, normalized)
#   12: high_band / close              (high-freq noise level, normalized)
#   13: wavelet_trend_slope            (trend direction & strength)
#   14: high_energy / (med_energy+1e-8)(noise-to-signal ratio)
#   15: spectral_entropy               (frequency complexity, 0=pure tone, 1=noise)
#
# Labels: soft sigmoid transform of future return instead of hard 0/1.

def soft_label(future_return: float, scale: float = SOFT_LABEL_SCALE,
               clip_lo: float = SOFT_LABEL_CLIP[0],
               clip_hi: float = SOFT_LABEL_CLIP[1]) -> float:
    """Convert an EXCESS return (return minus per-series drift) into a soft probability via sigmoid.
    excess=0     → 0.50 (maximally uncertain)
    excess=+0.5% → ~0.68 (moderately bullish, scale=150)
    excess=+2.0% → ~0.95 (very bullish, scale=150)
    excess=-1.0% → ~0.18 (very bearish, scale=150)
    """
    sig = 1.0 / (1.0 + np.exp(-future_return * scale))
    return float(np.clip(sig, clip_lo, clip_hi))


def build_features_for_series(candles: np.ndarray, window_size: int = 128) -> tuple:
    closes  = candles[:, 4]
    opens   = candles[:, 1]
    highs   = candles[:, 2]
    lows    = candles[:, 3]
    volumes = candles[:, 5]

    # ── Original indicators ──────────────────────────────────────────
    ema9_all   = ema(closes, 9)
    ema21_all  = ema(closes, 21)
    rsi14_all  = rsi(closes, 14)
    macd_line, _ = macd(closes, 12, 26, 9)
    bb_upper, bb_lower = bollinger(closes, 20, 2.0)

    # ── Wavelet decomposition (computed once on full series) ──────────
    wv = wavelet_decompose_series(closes)
    wv_trend   = wv['trend']
    wv_medium  = wv['medium']
    wv_high    = wv['high']
    wv_slope   = wavelet_trend_slope(wv_trend, window=10)
    wv_med_nrg = wavelet_energy_ratio(wv_medium, window=20)
    wv_hi_nrg  = wavelet_energy_ratio(wv_high, window=20)
    sp_entropy = spectral_entropy_rolling(closes, window=32)

    X_list, y_list = [], []
    n = len(candles)
    mean_bar_return = float(np.mean(np.diff(closes) / closes[:-1]))  # per-series drift

    for i in range(window_size, n - HORIZON):
        w_start = i - window_size
        w_end   = i

        ref_close  = closes[w_start]
        max_volume = np.max(volumes[w_start:w_end]) if np.max(volumes[w_start:w_end]) > 0 else 1.0
        if max_volume == 0:
            max_volume = 1.0

        # Normalize wavelet slope within the window for stability
        slope_window = wv_slope[w_start:w_end]
        slope_std = np.std(slope_window) + 1e-10

        rows = []
        for j in range(w_start, w_end):
            c = closes[j]
            prev_ret = 0.0 if j == w_start else (c - closes[j-1]) / closes[j-1]
            r9   = ema9_all[j]   if not np.isnan(ema9_all[j])   else c
            r21  = ema21_all[j]  if not np.isnan(ema21_all[j])  else c
            rsi_v = rsi14_all[j] if not np.isnan(rsi14_all[j])  else 50.0
            macd_v = macd_line[j] if not np.isnan(macd_line[j]) else 0.0
            bbu  = bb_upper[j]   if not np.isnan(bb_upper[j])   else c
            bbl  = bb_lower[j]   if not np.isnan(bb_lower[j])   else c

            # Wavelet features (safe division)
            med_e = wv_med_nrg[j] if wv_med_nrg[j] > 0 else 1e-10
            hi_e  = wv_hi_nrg[j]  if wv_hi_nrg[j] > 0  else 0.0

            rows.append([
                # ── Original 10 features ─────────────────────────────
                c / ref_close - 1,               # 0: normalized price
                volumes[j] / max_volume,         # 1: normalized volume
                rsi_v / 100.0,                   # 2: RSI [0,1]
                macd_v / c,                      # 3: MACD normalized
                prev_ret,                        # 4: previous return
                (highs[j] - lows[j]) / c,        # 5: range/close
                (r9 - c) / c,                    # 6: EMA9 distance
                (r21 - c) / c,                   # 7: EMA21 distance
                (bbu - c) / c,                   # 8: BB upper distance
                (bbl - c) / c,                   # 9: BB lower distance
                # ── Wavelet dual-path features (NEW) ─────────────────
                (wv_trend[j] - c) / c,           # 10: trend deviation
                wv_medium[j] / c,                # 11: medium-freq oscillation
                wv_high[j] / c,                  # 12: high-freq noise
                wv_slope[j] / slope_std,         # 13: trend slope (z-scored)
                hi_e / (med_e + 1e-10),          # 14: noise/signal energy ratio
                sp_entropy[j],                   # 15: spectral entropy [0,1]
            ])

        X_list.append(rows)

        # ── SOFT LABELS: sigmoid transform of future return ──────────
        anchor_close = closes[i]
        future = []
        for h in range(1, HORIZON + 1):
            future_return = (closes[i + h] - anchor_close) / anchor_close
            excess_return = future_return - mean_bar_return * h  # de-mean drift -> 0.5 = no edge
            future.append(soft_label(excess_return))
        y_list.append(future)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


def compute_volatility(candles: np.ndarray) -> np.ndarray:
    closes = candles[:, 4]
    n = len(closes)
    vol = np.zeros(n)
    for i in range(1, n):
        start = max(1, i - VOLATILITY_WINDOW)
        rets = (closes[start:i+1] - closes[start-1:i]) / closes[start-1:i]
        vol[i] = np.std(rets)
    return vol

print(f"Feature builder OK: {NUM_FEATURES} features, soft labels (scale={SOFT_LABEL_SCALE}).")
print(f"  Soft label examples:  +0.5% → {soft_label(0.005):.3f},  -0.5% → {soft_label(-0.005):.3f},  0.0% → {soft_label(0.0):.3f}")


Feature builder OK: 16 features, soft labels (scale=500).
  Soft label examples:  +0.5% → 0.924,  -0.5% → 0.076,  0.0% → 0.500


In [6]:
# ── Purged Embargoed Split ───────────────────────────────────────────────────

def purged_embargoed_split(X, y, horizon, embargo_frac, val_frac=0.1):
    total = len(X)
    val_size   = int(total * val_frac)
    val_start  = total - val_size
    embargo_sz = int(total * embargo_frac)
    train_end  = max(0, val_start - horizon - embargo_sz)
    purged = val_start - train_end
    return (
        X[:train_end], y[:train_end],
        X[val_start:],  y[val_start:],
        purged
    )

print("Split OK.")


Split OK.


In [7]:
# ── Download and preprocessing ───────────────────────────────────────────────

total_needed = HISTORICAL_CANDLES + BACKTEST_HOLDOUT

all_X_train, all_y_train = [], []
all_X_val,   all_y_val   = [], []
agent_candles = None

for sym in FORECASTER_SYMBOLS:
    print(f"\nDownloading {total_needed} candles for {sym}...")
    raw = fetch_candles(sym, total_needed, TIMEFRAME)
    train_raw = raw[:-BACKTEST_HOLDOUT] if len(raw) > BACKTEST_HOLDOUT else raw

    print(f"  {len(train_raw)} candles for training")

    min_needed = WINDOW_SIZE + HORIZON + 1
    if len(train_raw) < min_needed:
        print(f"  ! Insufficient data (minimum: {min_needed}) -- skipping.")
        continue

    if sym == AGENT_SYMBOL:
        agent_candles = train_raw.copy()

    X, y = build_features_for_series(train_raw, WINDOW_SIZE)
    Xtr, ytr, Xv, yv, purged = purged_embargoed_split(
        X, y, HORIZON, EMBARGO_FRACTION, VALIDATION_FRACTION
    )
    all_X_train.append(Xtr);  all_y_train.append(ytr)
    all_X_val.append(Xv);     all_y_val.append(yv)
    print(f"  train={len(Xtr)}, val={len(Xv)}, purged={purged}")

if not all_X_train:
    raise RuntimeError("No symbol returned data.")

if agent_candles is None:
    raise RuntimeError(f"No data for agent ({AGENT_SYMBOL}).")

X_train = np.concatenate(all_X_train, axis=0)
y_train = np.concatenate(all_y_train, axis=0)
X_val   = np.concatenate(all_X_val,   axis=0)
y_val   = np.concatenate(all_y_val,   axis=0)

print(f"\nTotal dataset -- train: {X_train.shape}, val: {X_val.shape}")


  -> BTC/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training


  train=31030, val=3486, purged=352



  -> ETH/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training


  train=31030, val=3486, purged=352



  -> SOL/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training


...

  -> XRP/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training


  train=31030, val=3486, purged=352



  -> DOGE/USDT: 36000 candles downloaded (requested: 36000)
  35000 candles for training


  train=31030, val=3486, purged=352



Total dataset -- train: (217210, 128, 16), val: (24402, 128, 16)


In [8]:
# ── Pre-compute agent features and volatility ────────────────────────────────

print("Pre-computing agent features...")
agent_X, _ = build_features_for_series(agent_candles, WINDOW_SIZE)
agent_vol  = compute_volatility(agent_candles)
agent_vol_samples = agent_vol[WINDOW_SIZE : len(agent_candles) - HORIZON]

print(f"Agent features: {agent_X.shape}, volatility: {agent_vol_samples.shape}")


Pre-computing agent features...


Agent features: (34868, 128, 16), volatility: (34868,)


In [9]:
# ── Multi-Head Self-Attention with Residual + LayerNorm (Keras 3) ────────────
# Replaces the old single-head SelfAttentionLayer.
#
# Why this is better:
#   - Multiple heads (4) capture different temporal patterns simultaneously
#     (trend, mean-reversion, breakout, consolidation)
#   - Residual connection prevents gradient degradation in deeper stacks
#   - LayerNorm stabilizes training and improves convergence speed
#   - Uses native Keras layer → no custom_objects needed for TF.js export
#
# We keep it wrapped in a single layer so build_bilstm stays clean.

class MultiHeadAttentionBlock(layers.Layer):
    """Multi-Head Attention + Add & Norm (Transformer encoder block, no FFN)."""

    def __init__(self, d_model: int, num_heads: int = 4,
                 dropout: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model   = d_model
        self.num_heads = num_heads
        self.drop_rate = dropout

    def build(self, input_shape):
        feat_dim = input_shape[-1]

        # Project input to d_model if dimensions don't match (for residual)
        self.needs_projection = (feat_dim != self.d_model)
        if self.needs_projection:
            self.input_proj = layers.Dense(self.d_model, name="mha_input_proj")

        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.d_model // self.num_heads,
            dropout=self.drop_rate,
            name="mha_core",
        )
        self.layernorm = layers.LayerNormalization(name="mha_layernorm")
        self.dropout   = layers.Dropout(self.drop_rate)
        super().build(input_shape)

    def call(self, x, training=None):
        residual = self.input_proj(x) if self.needs_projection else x
        attn_out = self.mha(query=x, value=x, key=x, training=training)
        attn_out = self.dropout(attn_out, training=training)
        return self.layernorm(residual + attn_out)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "d_model":   self.d_model,
            "num_heads":  self.num_heads,
            "dropout":    self.drop_rate,
        })
        return cfg

print("MultiHeadAttentionBlock OK (replaces SelfAttentionLayer).")


MultiHeadAttentionBlock OK (replaces SelfAttentionLayer).


In [10]:
# ── CNN 1D Multi-Scale → BiLSTM → Multi-Head Attention (CLASSIFICATION) ─────
#
# Architecture changes:
#   1. Conv1D multi-scale (kernels 3, 5, 7) extracts local candle patterns
#      (doji, engulfing, hammer) that raw LSTM misses
#   2. MaxPooling1D(2) halves sequence length → faster BiLSTM, less overfitting
#   3. MultiHeadAttentionBlock (4 heads) with residual + LayerNorm replaces
#      the old single-head SelfAttentionLayer
#
# The input/output shapes are unchanged:
#   Input:  (batch, WINDOW_SIZE, NUM_FEATURES)  →  (batch, 128, 10)
#   Output: (batch, HORIZON)                    →  (batch, 4)   [sigmoid]
#
# So the PPO agent, feature builder, and TF.js inference remain untouched.

# Tunable CNN parameters
CNN_FILTERS   = 32     # filters per kernel size
CNN_KERNELS   = [3, 5, 7]  # multi-scale: captures 3-bar, 5-bar, 7-bar patterns
CNN_POOL_SIZE = 2      # halves seq_len → BiLSTM sees (seq_len/2, concat_features)

def build_bilstm(seq_len=128, num_features=10, hidden=64,
                 dropout=0.2, horizon=4, l2_val=1e-5):
    reg = regularizers.L2(l2_val)
    inp = keras.Input(shape=(seq_len, num_features))

    # ── Stage 1: Conv1D multi-scale feature extraction ──────────────────
    # Each branch uses a different kernel size to capture patterns at
    # different temporal scales. All use 'same' padding so they output
    # the same seq_len and can be concatenated along the feature axis.
    conv_branches = []
    for k in CNN_KERNELS:
        branch = layers.Conv1D(
            filters=CNN_FILTERS, kernel_size=k,
            padding="same", activation="relu",
            kernel_regularizer=reg,
            name=f"conv1d_k{k}",
        )(inp)
        branch = layers.BatchNormalization(name=f"bn_k{k}")(branch)
        conv_branches.append(branch)

    # Concatenate: (batch, seq_len, CNN_FILTERS * len(CNN_KERNELS))
    # With defaults: (batch, 128, 96)
    x = layers.Concatenate(name="cnn_concat")(conv_branches)

    # Downsample: halves temporal resolution, reduces computation for BiLSTM
    # (batch, 128, 96) → (batch, 64, 96)
    x = layers.MaxPooling1D(pool_size=CNN_POOL_SIZE, name="cnn_pool")(x)
    x = layers.Dropout(dropout, name="cnn_dropout")(x)

    # ── Stage 2: BiLSTM temporal modeling ───────────────────────────────
    x = layers.Bidirectional(
        layers.LSTM(hidden, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        name="bilstm_1",
    )(x)
    x = layers.Dropout(dropout, name="bilstm1_dropout")(x)

    x = layers.Bidirectional(
        layers.LSTM(hidden // 2, return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        name="bilstm_2",
    )(x)
    x = layers.Dropout(dropout, name="bilstm2_dropout")(x)

    # ── Stage 3: Multi-Head Attention + Residual + LayerNorm ────────────
    # BiLSTM output dim = 2 * (hidden // 2) = hidden
    x = MultiHeadAttentionBlock(
        d_model=hidden, num_heads=4, dropout=dropout,
        name="mha_block",
    )(x)

    # ── Stage 4: Classification head ───────────────────────────────────
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(hidden, kernel_regularizer=reg,
                     bias_initializer=keras.initializers.Constant(0.01),
                     name="fc_hidden")(x)
    x = layers.LeakyReLU(negative_slope=0.01, name="fc_act")(x)
    x = layers.Dropout(dropout, name="fc_dropout")(x)

    # Sigmoid: independent P(up) per horizon step
    out = layers.Dense(horizon, activation="sigmoid", name="classifier_head")(x)

    return keras.Model(inputs=inp, outputs=out,
                       name="CNN_BiLSTM_MHA_Classifier")

with dist_scope():
    forecaster = build_bilstm(
        WINDOW_SIZE, NUM_FEATURES, HIDDEN_UNITS, DROPOUT, HORIZON, L2
    )

print(f"\nModel: {forecaster.name}")
print(f"  CNN branches: {len(CNN_KERNELS)} x {CNN_FILTERS} filters (kernels {CNN_KERNELS})")
print(f"  Pooling: {CNN_POOL_SIZE}x downsample → BiLSTM sees seq_len={WINDOW_SIZE // CNN_POOL_SIZE}")
print(f"  MHA: 4 heads, d_model={HIDDEN_UNITS}")
print(f"  Total params: {forecaster.count_params():,}")
forecaster.summary()



Model: CNN_BiLSTM_MHA_Classifier
  CNN branches: 3 x 32 filters (kernels [3, 5, 7])
  Pooling: 2x downsample → BiLSTM sees seq_len=64
  MHA: 4 heads, d_model=64
  Total params: 152,996


Model: "CNN_BiLSTM_MHA_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 16)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_k3 (Conv1D)  │ (None, 128, 32)   │      1,568 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_k5 (Conv1D)  │ (None, 128, 32)   │      2,592 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_k7 (Conv1D)  │ (None, 128, 32)   │      3,616 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_k3               │ (None, 128, 32)   │        128 │ conv1d_k3[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_k5               │ (None, 128, 32)   │        128 │ conv1d_k5[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_k7               │ (None, 128, 32)   │        128 │ conv1d_k7[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_concat          │ (None, 128, 96)   │          0 │ bn_k3[0][0],      │
│ (Concatenate)       │                   │            │ bn_k5[0][0],      │
│                     │                   │            │ bn_k7[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_pool            │ (None, 64, 96)    │          0 │ cnn_concat[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_dropout         │ (None, 64, 96)    │          0 │ cnn_pool[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 64, 128)   │     82,432 │ cnn_dropout[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm1_dropout     │ (None, 64, 128)   │          0 │ bilstm_1[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 64, 64)    │     41,216 │ bilstm1_dropout[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm2_dropout     │ (None, 64, 64)    │          0 │ bilstm_2[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mha_block           │ (None, 64, 64)    │     16,768 │ bilstm2_dropout[… │
│ (MultiHeadAttentio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 64)        │          0 │ mha_block[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_hidden (Dense)   │ (None, 64)        │      4,160 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_act (LeakyReLU)  │ (None, 64)        │          0 │ fc_hidden[0][0] 

 Total params: 152,996 (597.64 KB)

 Trainable params: 152,804 (596.89 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
# CNN-BiLSTM-MHA training — binary cross-entropy on SOFT labels + early stopping
# BCE works natively with soft targets in [0,1] — no code change needed.
# The model sees richer gradient signal: strong moves get steep gradients,
# ambiguous moves near 0.5 get shallow gradients (natural down-weighting).

class CosineAnnealingCallback(keras.callbacks.Callback):
    def __init__(self, lr_max, lr_min, total_epochs):
        super().__init__()
        self.lr_max = lr_max
        self.lr_min = lr_min
        self.total  = total_epochs

    def on_epoch_begin(self, epoch, logs=None):
        cosine = 0.5 * (1 + math.cos(math.pi * epoch / self.total))
        lr = self.lr_min + (self.lr_max - self.lr_min) * cosine
        self.model.optimizer.learning_rate.assign(lr)


# ── Metrics that actually work with SOFT labels ──────────────────────────────
# Keras' built-in "accuracy"/AUC expect binary y_true; our labels are soft floats
# in [0.02, 0.98], so they degenerate (AUC printed 0.0 every epoch). These binarize
# y_true at 0.5 on horizon 0 (the next-bar signal, == SIGNAL_HORIZON_INDEX on the TS side).
SIGNAL_INDEX = 0

class DirectionalAccuracy(keras.metrics.Metric):
    """% of bars where predicted direction (P(up)>0.5) matches the true soft label's direction."""
    def __init__(self, index=SIGNAL_INDEX, name="dir_acc", **kw):
        super().__init__(name=name, **kw)
        self.index = index
        self.total = self.add_weight(name="total", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")
    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.cast(y_true[:, self.index] > 0.5, tf.float32)
        yp = tf.cast(y_pred[:, self.index] > 0.5, tf.float32)
        match = tf.cast(tf.equal(yt, yp), tf.float32)
        self.total.assign_add(tf.reduce_sum(match))
        self.count.assign_add(tf.cast(tf.size(match), tf.float32))
    def result(self):
        return self.total / (self.count + 1e-8)
    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)

class BinaryAUCFromSoft(keras.metrics.AUC):
    """Standard AUC but binarizes the soft label (y_true>0.5) on horizon 0 before scoring."""
    def __init__(self, index=SIGNAL_INDEX, name="auc", **kw):
        super().__init__(name=name, **kw)
        self.index = index
    def update_state(self, y_true, y_pred, sample_weight=None):
        yt = tf.cast(y_true[:, self.index] > 0.5, tf.int32)
        yp = y_pred[:, self.index]
        return super().update_state(yt, yp, sample_weight)

with dist_scope():
    forecaster.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss="binary_crossentropy",
        metrics=[DirectionalAccuracy(name="dir_acc"), BinaryAUCFromSoft(name="auc")]
    )

callbacks = [
    CosineAnnealingCallback(LR, MIN_LR, FORECAST_EPOCHS),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=EARLY_STOP_PATIENCE,
        min_delta=1e-5, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        f"{WORK_DIR}/bilstm_best.keras",
        monitor="val_loss", save_best_only=True, verbose=1
    ),
]

print(f"Training BiLSTM (CLASSIFICATION) -- {FORECAST_EPOCHS} max epochs (batch={GLOBAL_BATCH})...")
print("Watch val_dir_acc (>0.50 = beats coin flip) and val_auc (>0.50 = real signal). "
      "Ignore val_loss magnitude: with soft labels its floor is ~0.69 regardless of skill.")
history = forecaster.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=FORECAST_EPOCHS,
    batch_size=GLOBAL_BATCH,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)
print("BiLSTM trained.")


In [12]:
# ── Pre-compute agent forecasts in batch ─────────────────────────────────────

CHUNK = 512
agent_forecasts = []
n_samples = len(agent_X)
print(f"Generating {n_samples} forecasts in batch...")
for start in range(0, n_samples, CHUNK):
    batch = agent_X[start : start + CHUNK]
    preds = forecaster.predict(batch, verbose=0)
    agent_forecasts.append(preds)
agent_forecasts = np.concatenate(agent_forecasts, axis=0)
print(f"Pre-computed forecasts: {agent_forecasts.shape}")


Generating 34868 forecasts in batch...


Pre-computed forecasts: (34868, 4)


## PPO — Reward V2 + Multi-GPU + Vectorization

Changes from the original pipeline:
- **Reward V2**: no idle penalty; profit bonus; forecaster-based opportunity cost; progressive hold-loser penalty
- **Entropy bonus** (0.01) on actor loss to prevent policy collapse
- **Vectorization**: 16 parallel episodes per update (~12x speedup over sequential)
- **Best checkpoint**: automatically saves the best model during training


In [13]:
# Reward V2 + helper functions (CLASSIFICATION inputs)

# Reward V2 hyperparameters
PROFIT_BONUS       = 3.0    # extra multiplier for profitable sells
OPPORTUNITY_COST   = 0.3    # weight of opportunity cost when flat
HOLD_WINNER_BONUS  = 1.5    # multiplier when holding a winning position
HOLD_LOSER_DECAY   = 0.05   # growing penalty for holding a losing position
LOSS_CUT_BONUS     = 0.001  # bonus for cutting a loss before stop-loss

def assemble_state(candles, i, forecast, position, entry_price, bars_in_pos, volatility):
    close = candles[i, 4]
    open_ = candles[i, 1]
    high  = candles[i, 2]
    low   = candles[i, 3]
    pnl   = (close - entry_price) / entry_price if position == 1 and entry_price > 0 else 0.0
    return [
        1 - position, pnl, min(bars_in_pos / 100.0, 1.0),
        (open_ - close) / close, (high - close) / close,
        (low - close) / close, (high - low) / close,
        position, volatility, *forecast[:4],
    ]

def apply_exit_guards(action, position, entry_price, current_close, sl, tp):
    if position != 1:
        return action
    pnl = (current_close - entry_price) / entry_price if entry_price > 0 else 0.0
    if pnl <= -sl or pnl >= tp:
        return 2
    return action

def compute_reward_v2(candles, i, action, position, entry_price, bars_in_pos, forecast):
    """Reward V2 - classification mode.

    forecast[0] is now P(up) in [0,1]. pred_strength is the signed
    directional confidence centered at 0 (positive = bullish view).
    """
    close      = candles[i, 4]
    next_close = candles[i + 1, 4]
    price_ret  = (next_close - close) / close
    pred_ret   = forecast[0]                  # probability in [0, 1]
    pred_strength = pred_ret - 0.5            # signed confidence in [-0.5, +0.5]

    # FLAT (no position)
    if position == 0:
        if action == 1:  # BUY
            base = price_ret - FEE_RATE - SLIPPAGE_PCT
            if pred_strength > 0:
                base += pred_strength * OPPORTUNITY_COST
            return base
        else:  # HOLD flat
            if pred_strength > PRED_STRENGTH_OPPORTUNITY_TH:
                return -pred_strength * OPPORTUNITY_COST * 0.3
            return 0.0

    # IN POSITION
    pnl = (close - entry_price) / entry_price if entry_price > 0 else 0.0

    if action == 2:  # SELL
        realized = pnl
        base = realized - 2 * FEE_RATE - 2 * SLIPPAGE_PCT
        if realized > 0:
            base += realized * PROFIT_BONUS
        elif realized < -LOSS_CUT_THRESHOLD and realized > -STOP_LOSS_PCT:
            base += LOSS_CUT_BONUS
        return base

    # HOLD in position
    if pnl > 0:
        return price_ret * HOLD_WINNER_BONUS
    else:
        time_factor = min(bars_in_pos / 20.0, 1.0)
        hold_penalty = -abs(pnl) * HOLD_LOSER_DECAY * time_factor
        return price_ret + hold_penalty

print("Reward V2 + helper functions OK (classification inputs).")
print(f"  Profit bonus: {PROFIT_BONUS}x | Opportunity cost: {OPPORTUNITY_COST}")
print(f"  Pred-strength opportunity threshold: {PRED_STRENGTH_OPPORTUNITY_TH:.3f} (P(up) > {0.5 + PRED_STRENGTH_OPPORTUNITY_TH:.2f})")

Reward V2 + helper functions OK (classification inputs).
  Profit bonus: 3.0x | Opportunity cost: 0.3
  Pred-strength opportunity threshold: 0.050 (P(up) > 0.55)


In [14]:
# ── Actor / Critic (inside strategy.scope, optimizers pre-built) ─────────────

def build_actor(state_size=13, action_size=3):
    inp = keras.Input(shape=(state_size,))
    x   = layers.Dense(128, activation="tanh")(inp)
    x   = layers.Dense(64,  activation="tanh")(x)
    out = layers.Dense(action_size, activation="softmax")(x)
    return keras.Model(inp, out, name="Actor")

def build_critic(state_size=13):
    inp = keras.Input(shape=(state_size,)
)
    x   = layers.Dense(128, activation="tanh")(inp)
    x   = layers.Dense(64,  activation="tanh")(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name="Critic")

with dist_scope():
    actor  = build_actor(STATE_SIZE, ACTION_SIZE)
    critic = build_critic(STATE_SIZE)
    policy_opt = keras.optimizers.Adam(POLICY_LR)
    value_opt  = keras.optimizers.Adam(VALUE_LR)
    policy_opt.build(actor.trainable_variables)
    value_opt.build(critic.trainable_variables)

update_count = 0

def get_current_lr(base_lr, decay, count):
    return max(base_lr * (decay ** count), PPO_MIN_LR)

def sample_action(probs):
    return np.random.choice(len(probs), p=probs)

def decide(state_vec):
    s = tf.convert_to_tensor([state_vec], dtype=tf.float32)
    probs = actor(s, training=False).numpy()[0]
    value = critic(s, training=False).numpy()[0, 0]
    action = sample_action(probs)
    log_prob = math.log(probs[action] + 1e-8)
    return action, log_prob, value

print("Actor/Critic OK (built inside strategy scope).")
actor.summary()

Actor/Critic OK (built inside strategy scope).


Model: "Actor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 13)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,243 (40.01 KB)

 Trainable params: 10,243 (40.01 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# ── PPO update functions with Batched Distributed Strategy ──
PPO_BATCH_SIZE = 8192
ENTROPY_COEFF = 0.01

def clip_grads(grads, max_norm):
    return [tf.clip_by_norm(g, max_norm) for g in grads]

@tf.function
def distributed_update_actor(states, actions, advantages, old_log_probs):
    def update_actor_step(st, ac, adv, olp):
        with tf.GradientTape() as tape:
            probs     = actor(st, training=True)
            log_probs = tf.math.log(probs + 1e-8)
            indices   = tf.stack([tf.range(tf.shape(ac)[0]), ac], axis=1)
            new_lp    = tf.gather_nd(log_probs, indices)
            ratio     = tf.exp(new_lp - olp)
            surr1     = ratio * adv
            surr2     = tf.clip_by_value(ratio, 1 - CLIP_RATIO, 1 + CLIP_RATIO) * adv
            policy_loss = -tf.reduce_mean(tf.minimum(surr1, surr2))
            entropy = -tf.reduce_mean(tf.reduce_sum(probs * log_probs, axis=1))
            loss = policy_loss - ENTROPY_COEFF * entropy
        grads = tape.gradient(loss, actor.trainable_variables)
        grads = clip_grads(grads, MAX_GRAD_NORM)
        policy_opt.apply_gradients(zip(grads, actor.trainable_variables))
        return loss, entropy
    
    per_replica_loss, per_replica_entropy = strategy.run(update_actor_step, args=(states, actions, advantages, old_log_probs))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_loss, axis=None), strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_entropy, axis=None)

@tf.function
def distributed_update_critic(states, returns):
    def update_critic_step(st, ret):
        with tf.GradientTape() as tape:
            values = tf.squeeze(critic(st, training=True), axis=1)
            loss   = tf.reduce_mean(tf.square(ret - values))
        grads = tape.gradient(loss, critic.trainable_variables)
        grads = clip_grads(grads, MAX_GRAD_NORM)
        value_opt.apply_gradients(zip(grads, critic.trainable_variables))
        return loss
    
    per_replica_loss = strategy.run(update_critic_step, args=(states, returns))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_loss, axis=None)

def compute_advantages_vectorized(rewards, values, terminals, gamma=0.99, gae_lambda=0.95):
    n_steps = rewards.shape[0]
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_adv = np.zeros(rewards.shape[1], dtype=np.float32)
    for t in reversed(range(n_steps)):
        if t == n_steps - 1:
            next_val = np.zeros(rewards.shape[1], dtype=np.float32)
        else:
            next_val = values[t+1]
        mask = 1.0 - terminals[t]
        delta = rewards[t] + gamma * next_val * mask - values[t]
        last_adv = delta + gamma * gae_lambda * mask * last_adv
        advantages[t] = last_adv
    return advantages

def compute_returns_vectorized(rewards, terminals, gamma=0.99):
    n_steps = rewards.shape[0]
    returns = np.zeros_like(rewards, dtype=np.float32)
    last_ret = np.zeros(rewards.shape[1], dtype=np.float32)
    for t in reversed(range(n_steps)):
        mask = 1.0 - terminals[t]
        last_ret = rewards[t] + gamma * mask * last_ret
        returns[t] = last_ret
    return returns

def ppo_update_batched(states_np, actions_np, log_probs_np, values_np, rewards_np, terminals_np):
    global update_count
    
    adv_np = compute_advantages_vectorized(rewards_np, values_np, terminals_np, GAMMA)
    ret_np = compute_returns_vectorized(rewards_np, terminals_np, GAMMA)
    
    flat_states = states_np.reshape(-1, states_np.shape[-1])
    flat_actions = actions_np.reshape(-1)
    flat_old_lp = log_probs_np.reshape(-1)
    flat_adv = adv_np.reshape(-1)
    flat_ret = ret_np.reshape(-1)
    
    dataset_size = flat_states.shape[0]
    indices = np.arange(dataset_size)
    
    p_loss_sum, p_ent_sum, v_loss_sum = 0.0, 0.0, 0.0
    steps = 0
    
    for _ in range(PPO_EPOCHS):
        np.random.shuffle(indices)
        for start in range(0, dataset_size, PPO_BATCH_SIZE):
            end = start + PPO_BATCH_SIZE
            idx_batch = indices[start:end]
            
            b_states = tf.constant(flat_states[idx_batch], dtype=tf.float32)
            b_actions = tf.constant(flat_actions[idx_batch], dtype=tf.int32)
            b_old_lp = tf.constant(flat_old_lp[idx_batch], dtype=tf.float32)
            b_adv = tf.constant(flat_adv[idx_batch], dtype=tf.float32)
            b_ret = tf.constant(flat_ret[idx_batch], dtype=tf.float32)
            
            l, e = distributed_update_actor(b_states, b_actions, b_adv, b_old_lp)
            vl = distributed_update_critic(b_states, b_ret)
            
            p_loss_sum += l.numpy()
            p_ent_sum += e.numpy()
            v_loss_sum += vl.numpy()
            steps += 1
            
    update_count += 1
    new_p_lr = get_current_lr(POLICY_LR, LR_DECAY, update_count)
    new_v_lr = get_current_lr(VALUE_LR,  LR_DECAY, update_count)
    policy_opt.learning_rate.assign(new_p_lr)
    value_opt.learning_rate.assign(new_v_lr)
    
    return p_loss_sum/steps, p_ent_sum/steps, v_loss_sum/steps

print(f"Distributed Batched PPO update OK. (batch_size={PPO_BATCH_SIZE}, entropy_coeff={ENTROPY_COEFF})")


Distributed Batched PPO update OK. (batch_size=8192, entropy_coeff=0.01)


In [16]:
# Vectorized episode runner V2 - CLASSIFICATION inputs

NUM_PARALLEL = 512

@tf.function
def tf_run_episodes_vectorized(closes, opens, highs, lows, forecasts, vol_samples, n_envs, n_steps):
    states_ta    = tf.TensorArray(tf.float32, size=n_steps)
    actions_ta   = tf.TensorArray(tf.int32,   size=n_steps)
    log_probs_ta = tf.TensorArray(tf.float32, size=n_steps)
    values_ta    = tf.TensorArray(tf.float32, size=n_steps)
    rewards_ta   = tf.TensorArray(tf.float32, size=n_steps)

    positions    = tf.zeros([n_envs], dtype=tf.int32)
    entry_prices = tf.zeros([n_envs], dtype=tf.float32)
    bars         = tf.zeros([n_envs], dtype=tf.int32)

    for idx in tf.range(n_steps):
        i = idx + WINDOW_SIZE

        close = closes[i]
        open_ = opens[i]
        high  = highs[i]
        low   = lows[i]
        vol   = vol_samples[idx]
        forecast = forecasts[idx]

        close_batch = tf.fill([n_envs], close)
        open_batch  = tf.fill([n_envs], open_)
        high_batch  = tf.fill([n_envs], high)
        low_batch   = tf.fill([n_envs], low)
        vol_batch   = tf.fill([n_envs], vol)
        forecast_batch = tf.tile(tf.expand_dims(forecast[:4], 0), [n_envs, 1])

        safe_entry = tf.where(entry_prices > 0.0, entry_prices, 1.0)
        pnls = tf.where((positions == 1) & (entry_prices > 0.0), (close_batch - safe_entry) / safe_entry, 0.0)

        st_1 = tf.cast(1 - positions, tf.float32)
        st_2 = pnls
        st_3 = tf.minimum(tf.cast(bars, tf.float32) / 100.0, 1.0)
        st_4 = (open_batch - close_batch) / close_batch
        st_5 = (high_batch - close_batch) / close_batch
        st_6 = (low_batch - close_batch) / close_batch
        st_7 = (high_batch - low_batch) / close_batch
        st_8 = tf.cast(positions, tf.float32)
        st_9 = vol_batch

        states = tf.concat([
            tf.expand_dims(st_1, 1), tf.expand_dims(st_2, 1), tf.expand_dims(st_3, 1),
            tf.expand_dims(st_4, 1), tf.expand_dims(st_5, 1), tf.expand_dims(st_6, 1),
            tf.expand_dims(st_7, 1), tf.expand_dims(st_8, 1), tf.expand_dims(st_9, 1),
            forecast_batch
        ], axis=1)

        probs  = actor(states, training=False)
        values = tf.squeeze(critic(states, training=False), axis=1)

        u = tf.random.uniform(tf.shape(probs), minval=0.00001, maxval=0.99999)
        actions = tf.argmax(tf.math.log(probs + 1e-8) - tf.math.log(-tf.math.log(u)), axis=-1, output_type=tf.int32)

        indices = tf.stack([tf.range(n_envs), actions], axis=1)
        log_probs = tf.math.log(tf.gather_nd(probs, indices) + 1e-8)

        force_sell = (positions == 1) & ((pnls <= -STOP_LOSS_PCT) | (pnls >= TAKE_PROFIT_PCT))
        effective_actions = tf.where(force_sell, 2, actions)

        next_close = closes[i + 1]
        price_ret  = (next_close - close) / close
        pred_ret   = forecast[0]                 # probability in [0, 1]
        pred_strength = pred_ret - 0.5           # signed confidence

        rewards = tf.zeros([n_envs], dtype=tf.float32)

        is_flat   = (positions == 0)
        flat_buy  = is_flat & (effective_actions == 1)
        flat_hold = is_flat & (effective_actions != 1)

        is_in     = (positions == 1)
        in_sell   = is_in & (effective_actions == 2)
        in_hold   = is_in & (effective_actions != 2)

        base_buy = price_ret - FEE_RATE - SLIPPAGE_PCT
        base_buy = tf.where(pred_strength > 0.0, base_buy + pred_strength * OPPORTUNITY_COST, base_buy)
        base_hold = tf.where(pred_strength > PRED_STRENGTH_OPPORTUNITY_TH, -pred_strength * OPPORTUNITY_COST * 0.3, 0.0)

        realized = pnls
        base_sell = realized - 2.0 * FEE_RATE - 2.0 * SLIPPAGE_PCT
        base_sell = tf.where(realized > 0.0, base_sell + realized * PROFIT_BONUS, base_sell)
        base_sell = tf.where((realized < -LOSS_CUT_THRESHOLD) & (realized > -STOP_LOSS_PCT), base_sell + LOSS_CUT_BONUS, base_sell)

        time_factor = tf.minimum(tf.cast(bars, tf.float32) / 20.0, 1.0)
        hold_penalty = -tf.abs(pnls) * HOLD_LOSER_DECAY * time_factor
        base_in_hold = tf.where(pnls > 0.0, price_ret * HOLD_WINNER_BONUS, price_ret + hold_penalty)

        rewards = tf.where(flat_buy, base_buy, rewards)
        rewards = tf.where(flat_hold, base_hold, rewards)
        rewards = tf.where(in_sell, base_sell, rewards)
        rewards = tf.where(in_hold, base_in_hold, rewards)

        positions = tf.where(flat_buy, 1, positions)
        positions = tf.where(in_sell,  0, positions)

        entry_prices = tf.where(flat_buy, close_batch, entry_prices)
        entry_prices = tf.where(in_sell,  0.0,         entry_prices)

        bars = tf.where(flat_buy, 0, bars)
        bars = tf.where(in_sell,  0, bars)
        bars = tf.where(in_hold,  bars + 1, bars)

        states_ta    = states_ta.write(idx, states)
        actions_ta   = actions_ta.write(idx, actions)
        log_probs_ta = log_probs_ta.write(idx, log_probs)
        values_ta    = values_ta.write(idx, values)
        rewards_ta   = rewards_ta.write(idx, rewards)

    return states_ta.stack(), actions_ta.stack(), log_probs_ta.stack(), values_ta.stack(), rewards_ta.stack()

def run_episodes_vectorized(candles, forecasts, vol_samples, n_envs=NUM_PARALLEL):
    n_steps = len(forecasts)
    closes = tf.constant(candles[:, 4], dtype=tf.float32)
    opens  = tf.constant(candles[:, 1], dtype=tf.float32)
    highs  = tf.constant(candles[:, 2], dtype=tf.float32)
    lows   = tf.constant(candles[:, 3], dtype=tf.float32)
    f_cast = tf.constant(forecasts, dtype=tf.float32)
    v_samp = tf.constant(vol_samples, dtype=tf.float32)

    st, ac, lp, va, rw = tf_run_episodes_vectorized(
        closes, opens, highs, lows, f_cast, v_samp,
        tf.constant(n_envs, dtype=tf.int32),
        tf.constant(n_steps, dtype=tf.int32)
    )

    st_np = st.numpy()
    ac_np = ac.numpy()
    lp_np = lp.numpy()
    va_np = va.numpy()
    rw_np = rw.numpy()

    terminals = np.zeros(n_steps, dtype=bool)
    terminals[-1] = True
    all_experiences = []
    for e in range(n_envs):
        all_experiences.append((st_np[:, e, :], ac_np[:, e], lp_np[:, e], va_np[:, e], rw_np[:, e], terminals))
    return all_experiences

print("Vectorized episode runner V2 OK (classification inputs).")

Vectorized episode runner V2 OK (classification inputs).


In [ ]:
# PPO V2 training loop with Distributed Multi-GPU Strategy + best checkpoint

import os
os.makedirs(f"{WORK_DIR}/models/ppo/policy", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models/ppo/value",  exist_ok=True)

TOTAL_UPDATES = 1000
EARLY_STOP_PATIENCE_PPO = 10      # stop after this many updates without a real improvement
MIN_DELTA_FRAC = 0.0025           # an update only counts as improvement if avg_reward beats
                                  # the best by >0.25% (relative). Filters out noise plateaus.

print(f"Training PPO V2 -- {TOTAL_UPDATES} updates x {NUM_PARALLEL} parallel episodes...")
print(f"Reward: no idle penalty | profit_bonus={PROFIT_BONUS} | entropy={ENTROPY_COEFF}")
print(f"Batched Mode: Distributed Multi-GPU Strategy | Batch: {PPO_BATCH_SIZE}")
print(f"Early stop after {EARLY_STOP_PATIENCE_PPO} updates without >{MIN_DELTA_FRAC:.2%} improvement.")
print()

best_avg_reward = -float('inf')
updates_since_best = 0

for update in range(TOTAL_UPDATES):
    t0 = time.time()
    
    # 1. Generate experiences for all 512 environments using the simulator
    st, ac, lp, va, rw = tf_run_episodes_vectorized(
        tf.constant(agent_candles[:, 4], dtype=tf.float32),
        tf.constant(agent_candles[:, 1], dtype=tf.float32),
        tf.constant(agent_candles[:, 2], dtype=tf.float32),
        tf.constant(agent_candles[:, 3], dtype=tf.float32),
        tf.constant(agent_forecasts, dtype=tf.float32),
        tf.constant(agent_vol_samples, dtype=tf.float32),
        tf.constant(NUM_PARALLEL, dtype=tf.int32),
        tf.constant(len(agent_forecasts), dtype=tf.int32)
    )
    
    st_np = st.numpy()
    ac_np = ac.numpy()
    lp_np = lp.numpy()
    va_np = va.numpy()
    rw_np = rw.numpy()
    
    terminals = np.zeros((len(agent_forecasts), NUM_PARALLEL), dtype=bool)
    terminals[-1, :] = True
    
    # 2. Batched GPU Update
    pl, pe, vl = ppo_update_batched(st_np, ac_np, lp_np, va_np, rw_np, terminals)
    
    elapsed = time.time() - t0
    avg_reward = np.mean(np.sum(rw_np, axis=0))
    avg_buys   = np.mean(np.sum(ac_np == 1, axis=0))
    avg_sells  = np.mean(np.sum(ac_np == 2, axis=0))
    
    # Count as improvement only if it beats the best by more than MIN_DELTA_FRAC
    # (relative). Floored at an absolute 1.0 so a near-zero best can not make the
    # threshold collapse to 0. First update (best == -inf) always counts.
    if best_avg_reward == -float('inf'):
        improved = True
    else:
        threshold = best_avg_reward + max(abs(best_avg_reward), 1.0) * MIN_DELTA_FRAC
        improved = avg_reward > threshold

    if improved:
        best_avg_reward = avg_reward
        updates_since_best = 0
        marker = " * BEST"
        actor.save(f"{WORK_DIR}/models/ppo/policy/best_model.keras")
        critic.save(f"{WORK_DIR}/models/ppo/value/best_model.keras")
    else:
        updates_since_best += 1
        marker = ""
        
    plateau_warn = ""
    if updates_since_best >= EARLY_STOP_PATIENCE_PPO:
        plateau_warn = f" [PLATEAU {updates_since_best}/{EARLY_STOP_PATIENCE_PPO}]"
        
    print(f"  Update {update+1:02d}/{TOTAL_UPDATES} | avg_reward={avg_reward:+.4f} | "
          f"buys={avg_buys:.1f} sells={avg_sells:.1f} | {elapsed:.0f}s{marker}{plateau_warn}")

    # 3. Early stop: plateau reached, no real improvement for PATIENCE updates
    if updates_since_best >= EARLY_STOP_PATIENCE_PPO:
        print()
        print(f"  Early stopping at update {update+1}: no >{MIN_DELTA_FRAC:.2%} "
              f"improvement in {EARLY_STOP_PATIENCE_PPO} updates. Best checkpoint kept.")
        break

print()
print(f"PPO V2 trained. Best avg_reward: {best_avg_reward:+.4f}")

In [ ]:
# ── Save final models (PPO uses best checkpoint) ─────────────────────────────

import os, shutil
os.makedirs(f"{WORK_DIR}/models/bilstm", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models/ppo/policy", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models/ppo/value",  exist_ok=True)

# BiLSTM: save current state (already has best weights via restore_best_weights)
forecaster.save(f"{WORK_DIR}/models/bilstm/model.keras")

# PPO: use the best checkpoint saved during training
shutil.copy(f"{WORK_DIR}/models/ppo/policy/best_model.keras",
            f"{WORK_DIR}/models/ppo/policy/model.keras")
shutil.copy(f"{WORK_DIR}/models/ppo/value/best_model.keras",
            f"{WORK_DIR}/models/ppo/value/model.keras")

print(f"Models saved in {WORK_DIR}/models/")
print("  bilstm/model.keras")
print("  ppo/policy/model.keras  (best checkpoint)")
print("  ppo/value/model.keras   (best checkpoint)")

## Export to TensorFlow.js

All three models are exported as **`tfjs_graph_model`** (frozen computation graph).
This avoids the Keras 3 layers-model format that `tfjs-layers` (v4.x) cannot deserialize
(error: `Corrupted configuration, expected array for nodeData`).

Graph models are inference-only, which is fine here — training happens on Kaggle, the
bot only runs `predict()` in Node.

The final `tfjs_models.zip` contains everything needed to run the bot:
```
tfjs_models/
  bilstm/model.json        <- graph model (MultiHeadAttentionBlock baked in)
  ppo/
    policy/model.json      <- actor (graph model)
    value/model.json       <- critic (graph model)
```
**Setup:** download `tfjs_models.zip` from the Kaggle output panel and extract into `./models/`.


In [ ]:
# Install the TFJS converter HERE (not at the top): its deps can break the CUDA
# context, but by this point all GPU training/building is finished. --no-deps
# keeps TensorFlow untouched; the SavedModel->graph converter needs only TF + hub.
!pip install -q --no-deps tensorflowjs
!pip install -q --no-deps tensorflow-hub

# ── Export to TensorFlow.js (all models as graph_model) ──────────────────────
import os
import shutil, subprocess

OUTPUT_BASE = f"{WORK_DIR}/tfjs_models"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# ── BiLSTM: rebuild on CPU and export as SavedModel ─────────────────────────
# The new architecture uses only native Keras layers (Conv1D, BiLSTM,
# MultiHeadAttention, LayerNorm) so NO custom_objects are needed.
# We still need to re-build on CPU to avoid CudnnRNN ops in the SavedModel.

cpu_keras_path = f"{WORK_DIR}/models/bilstm/cpu_model.keras"
cpu_saved_path = f"{WORK_DIR}/models/bilstm/cpu_saved_model"

# Rebuild the exact same architecture on CPU, copy weights
cpu_forecaster = build_bilstm(WINDOW_SIZE, NUM_FEATURES, HIDDEN_UNITS, DROPOUT, HORIZON, L2)
cpu_forecaster.set_weights(forecaster.get_weights())
cpu_forecaster.save(cpu_keras_path)
print("BiLSTM saved as .keras (CPU copy)")

# Inline script runs conversion without GPU to avoid CudnnRNN in the SavedModel.
# NOTE: MultiHeadAttentionBlock is a custom layer, so we must register it.
convert_script = f"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import math, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

class MultiHeadAttentionBlock(layers.Layer):
    def __init__(self, d_model, num_heads=4, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model   = d_model
        self.num_heads = num_heads
        self.drop_rate = dropout
    def build(self, input_shape):
        feat_dim = input_shape[-1]
        self.needs_projection = (feat_dim != self.d_model)
        if self.needs_projection:
            self.input_proj = layers.Dense(self.d_model, name='mha_input_proj')
        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads, key_dim=self.d_model // self.num_heads,
            dropout=self.drop_rate, name='mha_core')
        self.layernorm = layers.LayerNormalization(name='mha_layernorm')
        self.dropout   = layers.Dropout(self.drop_rate)
        super().build(input_shape)
    def call(self, x, training=None):
        residual = self.input_proj(x) if self.needs_projection else x
        attn_out = self.mha(query=x, value=x, key=x, training=training)
        attn_out = self.dropout(attn_out, training=training)
        return self.layernorm(residual + attn_out)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({{'d_model': self.d_model, 'num_heads': self.num_heads, 'dropout': self.drop_rate}})
        return cfg

model = keras.models.load_model('{cpu_keras_path}', custom_objects={{'MultiHeadAttentionBlock': MultiHeadAttentionBlock}})
model.export('{cpu_saved_path}')
print('SavedModel exported on CPU (no CudnnRNN)')
"""

with open('/tmp/convert_bilstm.py', 'w') as f:
    f.write(convert_script)

result = subprocess.run(
    ['python', '/tmp/convert_bilstm.py'],
    capture_output=True, text=True,
    env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''}
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

# Convert BiLSTM SavedModel -> tfjs_graph_model
result2 = subprocess.run([
    'tensorflowjs_converter',
    '--input_format=tf_saved_model',
    '--output_format=tfjs_graph_model',
    '--signature_name=serving_default',
    cpu_saved_path,
    f'{OUTPUT_BASE}/bilstm'
], capture_output=True, text=True,
   env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(result2.stdout)
if result2.returncode != 0:
    print('STDERR:', result2.stderr[-500:])
else:
    print('BiLSTM converted to tfjs_graph_model')
    # --- PATCH: Remove control dependencies from TF.js model.json ---
    model_json_path = f'{OUTPUT_BASE}/bilstm/model.json'
    with open(model_json_path, 'r') as f:
        m_data = json.load(f)
    for node in m_data.get('modelTopology', {}).get('node', []):
        if 'input' in node:
            node['input'] = [i for i in node['input'] if not i.startswith('^')]
    with open(model_json_path, 'w') as f:
        json.dump(m_data, f)
    print('Control dependencies stripped from BiLSTM model.json')

# ── Actor / Critic: SavedModel -> tfjs_graph_model ───────────────────────────

def export_keras_to_tfjs_graph(model_obj, name):
    saved_path = f"{WORK_DIR}/models/ppo/{name}/saved_model"
    if os.path.exists(saved_path):
        shutil.rmtree(saved_path)
    model_obj.export(saved_path)

    out_path = f"{OUTPUT_BASE}/ppo/{name}"
    res = subprocess.run([
        'tensorflowjs_converter',
        '--input_format=tf_saved_model',
        '--output_format=tfjs_graph_model',
        '--signature_name=serving_default',
        saved_path,
        out_path
    ], capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(f'STDERR ({name}):', res.stderr[-500:])
        return False
    print(f'  {name} -> tfjs_graph_model OK')
    return True

export_keras_to_tfjs_graph(actor,  'policy')
export_keras_to_tfjs_graph(critic, 'value')

# Sanity check
for sub in ['bilstm', 'ppo/policy', 'ppo/value']:
    with open(f'{OUTPUT_BASE}/{sub}/model.json', 'r') as f:
        fmt = json.load(f).get('format', '<missing>')
    print(f'  {sub}: format = {fmt}')
    assert fmt == 'graph-model', f'{sub} expected graph-model, got {fmt}'

shutil.make_archive(f'{WORK_DIR}/tfjs_models', 'zip', OUTPUT_BASE)
print(f'\nFinal file: {WORK_DIR}/tfjs_models.zip')
subprocess.run(['find', OUTPUT_BASE, '-type', 'f'], check=True)


In [ ]:
# ── Validation Visualization ─────────────────────────────────────────
# Selects 3 random scenarios from the validation set to
# compare the model's prediction (red line) with what actually
# happened (green line, actual soft labels).

import matplotlib.pyplot as plt
import numpy as np

print("Generating validation charts...")
# Choose 3 different scenarios from the Validation Set
np.random.seed(42)
indices = np.random.choice(len(X_val), 3, replace=False)

y_pred = forecaster.predict(X_val[indices])

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Model Validation: 3 Scenarios (Prediction vs. Reality)', fontsize=18)

for i, idx in enumerate(indices):
    ax = axes[i]
    # Price history (Normalized) for a 128-period window.
    price_history = X_val[idx, :, 0]
    
    # History plot
    ax.plot(range(len(price_history)), price_history, label='Historical Price (Normal)', color='blue', linewidth=1.5)
    
    # X-axis for the future (Predictive horizon)
    start_future = len(price_history)
    future_x = range(start_future, start_future + HORIZON)
    
    actual_labels = y_val[idx]
    pred_labels = y_pred[i]
    
    # Second Y-axis for probabilities (Soft Labels at [0, 1])
    ax2 = ax.twinx()
    ax2.plot(future_x, actual_labels, marker='o', markersize=6, label='Reality (Soft Label)', color='green', linestyle='--')
    ax2.plot(future_x, pred_labels, marker='X', markersize=8, label='Model Prediction', color='red', linestyle='-')
    
    # Formatting and Captions
    ax.set_title(f'Scenario {i+1} (Validation Sample #{idx})', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel('Standardized Price')
    ax2.set_ylabel('P(High) - Soft Label')
    ax2.set_ylim(-0.05, 1.05)
    
    # Neutral line of 0.5 (no strong trend)
    ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.6)

    # Unifying captions from both axes
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

## Loading in TypeScript

Kaggle notebook: [https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading](https://www.kaggle.com/code/acaurangel/bilstm-ppo-self-attention-ai-spot-trading)

**Setup:**
1. Download `tfjs_models.zip` from the Kaggle output panel after training.
2. Extract into `./models/` at the project root.

**Model formats:** all three are `tfjs_graph_model`, loaded with `tf.loadGraphModel`.

```typescript
const forecaster = await tf.loadGraphModel('file://./models/bilstm/model.json');
const actor      = await tf.loadGraphModel('file://./models/ppo/policy/model.json');
const critic     = await tf.loadGraphModel('file://./models/ppo/value/model.json');
```